# Turn reproducible evidence into a clear manuscript

This notebook maps the completed experiments to the new draft03, verifies central numbers, and distinguishes measured results from proposed extensions.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/v2_data.py').is_file()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/v2_data.py').is_file())
sys.path.insert(0, str(PROJECT))
REPOSITORY = PROJECT.parents[1]
from worldmodels.v2_data import load_cohort
from worldmodels import v2_evaluation as ev
from worldmodels import v2_models as models
from worldmodels.v2_runner import window_errors, verify_freeze
COHORT = PROJECT/'artifacts/runs/reference-v2-cohort-v1'
RUN = PROJECT/'artifacts/runs/reference-v2-core-v1'
TRANSFER = PROJECT/'artifacts/runs/reference-v2-transfer-v1'
FIGURES = PROJECT/'artifacts/executed-reference-v2/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
USED_INPUTS={}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def record_input(path):
    path=Path(path).resolve()
    key=str(path.relative_to(REPOSITORY)) if path.is_relative_to(REPOSITORY) else str(path)
    USED_INPUTS[key]=sha(path)
def read_json(path):
    path = Path(path)
    assert path.is_file(), f'Required completed artifact is absent: {path}. Read docs/reference_v2/README.md for prerequisites.'
    record_input(path)
    return json.loads(path.read_text())
def read_csv(path, **kwargs):
    record_input(path)
    return pd.read_csv(path, **kwargs)
def artifact(name, directory=RUN):
    path = directory/name
    manifest = read_json(directory/'artifact_hashes.json')
    assert name in manifest and path.is_file(), f'Unrecorded or missing artifact: {path}'
    assert sha(path) == manifest[name], f'Artifact hash differs from the completed run: {path}'
    record_input(path)
    return path
def completed_run():
    result = read_json(artifact('results.json'))
    assert result['status']=='complete' and not result.get('test_only', False)
    return result
def save_figure(figure, name):
    figure.savefig(FIGURES/(name+'.svg'), bbox_inches='tight')
    figure.savefig(FIGURES/(name+'.png'), dpi=150, bbox_inches='tight')
freeze = verify_freeze()
record_input(PROJECT/'docs/reference_v2/protocol_freeze.json')
x, valid, metadata, actual_times = load_cohort(COHORT, verify=True)
cohort_identity=read_json(COHORT/'identity.json');record_input(COHORT/'cohort.csv')
for name in cohort_identity['prepared_inputs']:
    record_input(REPOSITORY/cohort_identity['prepared_directory']/name)
assert x.shape == (len(metadata),24,16,2) and len(metadata)==2815
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
print('Verified real observations:',len(metadata),'windows;',metadata.source_id.nunique(),'source videos.')
print('Observation units: pelvis-relative 2D coordinates divided by a prefix torso scale. Times are decoder seconds.')


## Trace the study's measured claims

A manuscript claim needs an observable result and an exact input identity. A proposed extension has a different evidence state. We read the new draft's claim ledger alongside the completed core and external reports so the paper cannot silently turn a design into a measured improvement.

In [ ]:
result=completed_run()
manuscript=PROJECT/'docs/draft03'
claims=read_json(manuscript/'claim_ledger.json')
assert isinstance(claims,dict) and isinstance(claims['claims'],list)
print('Manuscript evidence identity:',claims['identity'])
display(pd.DataFrame(claims['claims'])[['id','claim','artifact','scope']])
print('The preserved previous manuscript is docs/draft02. This notebook reads only the new draft03.')


## Check counts, directions, and comparisons

Every displayed number should identify the population and comparison that produced it. The primary internal anomaly contrast is calibrated posture-plus-velocity versus low motion, and the posture-only calibrated detector is a required secondary control. Forecast improvement and anomaly ranking are separate claims.

In [ ]:
source=read_csv(artifact('source_anomaly_scores.csv'))
assert result['windows']==len(metadata) and result['sources']==metadata.source_id.nunique()
assert result['normal_sources']==metadata.loc[metadata.label.eq('normal'),'source_id'].nunique()
np.testing.assert_allclose(source.low_motion,-source.persistence)
computed=ev.within_fold_summary(source,'calibrated_two_sided')['pair_weighted_auroc']
np.testing.assert_allclose(computed,result['primary_anomaly']['candidate_auroc'],atol=1e-12,rtol=0)
display(pd.DataFrame(result['forecast_contrasts']).query("method=='ridge_posture_velocity'"))
display({name:result[name] for name in ['primary_anomaly','posture_calibration_contrast']})


## Rebuild every manuscript table from its source results

The publication tables are formatted views of the completed runs. We reconstruct all six tables from source-level summaries and cohort memberships, then compare every cell with the manuscript CSV. The renderer supplies names and decimal formatting, while the numerical values come from the declared evidence. This computation writes only into the notebook's own output area.

In [ ]:
import importlib.util
renderer_path=PROJECT/'scripts/build_next_draft03.py';record_input(renderer_path)
sys.path.insert(0,str(PROJECT/'scripts'))
specification=importlib.util.spec_from_file_location('medwm_publication_renderer',renderer_path)
renderer=importlib.util.module_from_spec(specification);specification.loader.exec_module(renderer)
transfer=read_json(artifact('results.json',TRANSFER))
external_windows=read_csv(artifact('window_metadata.csv',TRANSFER))
mean_priors=read_json(artifact('results.json',PROJECT/'artifacts/runs/reference-v2-mean-prior-posthoc-v1'))
forecasts={row['arm']:row for row in result['normal_forecast'] if row['horizon']==8}
for row in mean_priors['comparisons']:
    if row['reference']=='persistence':
        forecasts[row['method']]={'mse':row['method_mse'],'ci95':row['method_ci95']}
ranks={row['score']:row for row in result['rankings']}
forecast_arms=['persistence','damped_velocity','own','bilateral','static','ridge_posture','ridge_posture_velocity','untrained_probe','mean_pose','mean_residual']
anomaly_arms=['calibrated_two_sided','posture_two_sided','low_motion','one_sided','marginal_two_sided','posture_distance','motion_quality_distance','negative_prefix_motion','high_prefix_missingness','excess_bilateral','excess_ridge_posture_velocity']
def forecast_row(arm):
    row=forecasts[arm]
    return [renderer.name(arm),renderer.fmt(row['mse']*1000,2),renderer.interval(np.array(row['ci95'])*1000,2)]
table_rows={
    'cohort':[[label,str(group.source_id.nunique()),str(group.sequence_id.nunique()),str(len(group))] for label,group in metadata.groupby('label')],
    'external_cohort':[[label,str(group.participant_id.nunique()),str(group.recording_id.nunique()),str(len(group))] for label,group in external_windows.groupby('label')],
    'forecasts':[forecast_row(arm) for arm in forecast_arms],
    'complete_forecasts':[forecast_row(arm) for arm in forecasts],
    'rankings':[[renderer.SCORES[arm],renderer.fmt(ranks[arm]['within_fold']['pair_weighted_auroc']),renderer.fmt(ranks[arm]['pooled']['auroc']),renderer.fmt(ranks[arm]['pooled']['average_precision'])] for arm in anomaly_arms],
    'external_rankings':[[renderer.name(arm),renderer.fmt(row['auroc']),renderer.interval(row['auroc_ci95']),renderer.fmt(row['average_precision'])] for arm,row in transfer['anomaly_scores'].items()],
}
rebuild=PROJECT/'artifacts/executed-reference-v2/manuscript_rebuild';rebuild.mkdir(parents=True,exist_ok=True)
table_checks=[]
for name,rows in table_rows.items():
    published=read_csv(manuscript/(name+'.csv'),dtype=str)
    rebuilt=pd.DataFrame(rows,columns=published.columns).astype(str)
    pd.testing.assert_frame_equal(rebuilt,published)
    rebuilt.to_csv(rebuild/(name+'.csv'),index=False)
    table_checks.append({'table':name,'rows':len(rebuilt),'all_cells_match':True})
display(pd.DataFrame(table_checks))


## Keep the evidence states visible

Successful software checks, fitted models, evaluated predictions, and proposed methods are different accomplishments. The execution ledger records what actually ran. We retain an exact copy of the coordinator's ledger at execution time because its administrative acceptance status can change afterward. A latent or stochastic model must not appear as a measured contribution merely because its illustration or mathematical idea was described.

In [ ]:
ledger_bytes=(PROJECT/'docs/reference_v2/run_record.json').read_bytes()
ledger_digest=hashlib.sha256(ledger_bytes).hexdigest()
ledger_snapshot=PROJECT/'artifacts/executed-reference-v2/inputs'/f'coordinator_ledger_{ledger_digest}.json'
ledger_snapshot.parent.mkdir(parents=True,exist_ok=True)
if ledger_snapshot.exists():
    assert ledger_snapshot.read_bytes()==ledger_bytes
else:
    ledger_snapshot.write_bytes(ledger_bytes)
record=read_json(ledger_snapshot)
print('Exact coordinator ledger snapshot:',ledger_digest)
request=read_json(artifact('request.json'));configuration=request['config']
ledger=read_json(artifact('model_fit_ledger.json'))
available=set(read_csv(artifact('refs_source_scores.csv')).arm) | set(read_csv(artifact('masks_source_scores.csv')).arm)
registry=[]
references=['persistence','damped_velocity','own','bilateral','shifted','static','ridge_posture','ridge_posture_velocity']
if configuration['untrained_probe']: references.append('untrained_probe')
for name in references:
    analytic=name in ['persistence','damped_velocity']
    registry.append({'candidate':name,'family':'reference','declared':True,'fitting':'analytic reference' if analytic else 'fitted normal-only model',
                     'status':'evaluated' if name in available else 'MISSING from completed evaluation; inspect run ledger'})
for policy in configuration['policies']:
    for seed in configuration['seeds']:
        name=f'mlp_{policy}_seed{seed}'
        roles=[f'outer{fold}:mask_{policy}:seed{seed}' for fold in range(5)]
        fitted=sum(any(role in item['roles'] for item in ledger.values()) for role in roles)
        registry.append({'candidate':name,'family':'mask policy and optimization seed','declared':True,'fitting':f'{fitted}/5 outer refits recorded',
                         'status':'evaluated' if name in available else 'MISSING from completed evaluation; inspect run ledger'})
    name=f'mlp_{policy}'
    registry.append({'candidate':name,'family':'fixed seed-mean prediction','declared':True,'fitting':'mean of the three declared coordinate predictions',
                     'status':'evaluated' if name in available else 'MISSING from completed evaluation; inspect run ledger'})
if 'motion_balanced' in configuration['policies']:
    registry.append({'candidate':'mlp_pattern_only','family':'mask-pattern residual diagnostic','declared':True,'fitting':'normal-only seed27',
                     'status':'evaluated' if 'mlp_pattern_only' in available else 'MISSING from completed evaluation; inspect run ledger'})
for name in ['calibration_models.json','posture_calibration_models.json']:
    models_record=read_json(artifact(name))
    registry.append({'candidate':name,'family':'normal-reference calibration ensemble','declared':True,'fitting':f'{len(models_record)}/15 components recorded',
                     'status':'evaluated' if len(models_record)==15 else 'INCOMPLETE; inspect run ledger'})
mean_priors=read_json(artifact('results.json',PROJECT/'artifacts/runs/reference-v2-mean-prior-posthoc-v1'))
for name in ['mean_pose','mean_residual']:
    registry.append({'candidate':name,'family':'separately declared post hoc mean-prior control','declared':True,
                     'fitting':'five untuned normal-only outer fits','status':mean_priors['status']})
with pd.option_context('display.max_rows',None,'display.max_colwidth',None):
    display(pd.DataFrame(registry))
    display(pd.DataFrame([{'score':item['score'],'status':'evaluated with its declared direction',
                           'within_fold_AUROC':item['within_fold']['pair_weighted_auroc']}
                          for item in result['rankings']]))
floor_sensitivity=read_json(artifact('results.json',PROJECT/'artifacts/runs/reference-v2-floor-posthoc-v2'))
display({'separate_posthoc_analysis':floor_sensitivity['identity'],'status':floor_sensitivity['status'],
         'frozen_floors':floor_sensitivity['floors'],'forecast_refit':floor_sensitivity['forecasts_refit']})
display(pd.DataFrame([{'floor':row['floor'],'maximum_score_change':row['max_abs_score_change_vs_1e8'],
                       'changed_rank_pairs':row['ranking_agreement_vs_1e8']['changed_pair_directions'],
                       'within_fold_AUROC':row['primary_anomaly']['candidate_auroc']}
                      for row in floor_sensitivity['results']]))
print(floor_sensitivity['serialization'])
print('The failed first floor replay is preserved under its original identity. Its guard exposed an omitted CSV round trip; the revised sensitivity reproduces the exact producer path without altering the frozen scores.')
failed_floor=read_json(PROJECT/'artifacts/runs/reference-v2-floor-posthoc-v1/failure.json')
display({'preserved_attempt':failed_floor['identity'],'status':failed_floor['status'],'cause':failed_floor['cause']})
assert 'extension_gates' in record,'The final run ledger must state the extension-gate decisions.'
gates=record['extension_gates']
display(pd.DataFrame.from_dict(gates,orient='index') if isinstance(gates,dict) else pd.DataFrame(gates))
display(record)
print('Notebook23 includes an explicitly labeled real training demonstration. Its training error is absent from the held-out result table.')
print('Read the resource/extension gate record before describing deferred models as implemented or evaluated.')


## Verify the manuscript evidence and inspect its figures

Draft03 has editable prose and a dedicated generator. Its evidence record links numerical tables and figures to completed runs. Read figures at full size and check their captions against units, sample counts, score direction, and uncertainty. A strong contribution should be understandable without specialist vocabulary.

In [ ]:
evidence=read_json(manuscript/'evidence.json')
print('Evidence identity:',evidence['identity'])
display(pd.DataFrame.from_dict(evidence['counts'],orient='index',columns=['count']))
display(evidence['displayed_values'])
display(evidence['inputs'])
print('Manuscript PDF exists:',(manuscript/'manuscript.pdf').is_file())
artifact('refs_predictions.npz');artifact('source_scores.csv')
renderer.OUT=rebuild;(rebuild/'figures').mkdir(exist_ok=True)
with plt.rc_context():
    plt.rcdefaults()
    renderer.figures(result,transfer,forecasts,ranks,metadata)
from IPython.display import SVG
figure_checks=[]
for figure in sorted((manuscript/'figures').glob('*.svg')):
    record_input(figure)
    regenerated=rebuild/'figures'/figure.name
    assert regenerated.is_file() and sha(regenerated)==sha(figure),f'The rebuilt figure differs: {figure.name}'
    figure_checks.append({'figure':figure.name,'sha256':sha(figure),'exact_svg_match':True})
    display(SVG(filename=str(regenerated)))
assert len(figure_checks)==5 and len(table_checks)==6
display(pd.DataFrame(figure_checks))
(PROJECT/'docs/reference_v2/notebook_26_publication_rebuild.json').write_text(json.dumps({'tables':table_checks,'figures':figure_checks,'renderer_sha256':sha(renderer_path)},indent=2)+'\n')
print('Read the independent final review for exact accepted artifact identities after all material edits.')


## Explain the contribution in complete sentences

The last check is conceptual. Explain the observation model, the role of its simple reference, the strongest measured finding, and the remaining generalization requirement in four connected sentences. The conclusion must match the verified comparison, including the strongest simple alternative. It should help a reader understand what has been learned and why that finding matters for future medical world models.

In [ ]:
print('Evidence paths:')
for path in [RUN/'results.json',TRANSFER/'results.json',manuscript/'manuscript.pdf',PROJECT/'docs/reference_v2/protocol.md']:
    record_input(path)
    print(path.relative_to(PROJECT),sha(path))
print('The manuscript generator and independent review, rather than this notebook, determine the final publication claim.')
forecast=next(row for row in result['forecast_contrasts'] if row['method']=='ridge_posture' and row['reference']=='persistence')
print('A plain-language explanation grounded in the completed evidence follows.')
print('The study predicts the next eight leg observations from body-relative joint positions estimated in short walking videos.')
print(f"Across {result['normal_sources']} normal source videos, a small current-posture model reduced mean squared forecast error from {forecast['reference_mse']:.4f} to {forecast['method_mse']:.4f} relative to repeating the last pose.")
print('The tested structured masks and normal-reference anomaly score did not improve their strongest declared simple comparators, showing why forecast quality and abnormality ranking require separate tests.')
print('The reproducible source-grouped benchmark and frozen external transfer provide a tested movement-observation component, while independent clinical validation remains necessary for patient-level use.')

# Retain the exact inputs used by this independent-kernel execution.
input_path=PROJECT/'artifacts/executed-reference-v2/inputs/26_manuscript_evidence_and_claims.json'
input_path.parent.mkdir(parents=True,exist_ok=True)
input_path.write_text(json.dumps({'path_base':'repository root','inputs':USED_INPUTS},indent=2)+'\n')
